# Module 1 — The Machine Learning Pipeline & Tools Refresher
### Classroom walkthrough: raw data → trained model → live web app

**What we build:**  
A model that predicts whether a customer is a **Low / Medium / High** spender  
based on demographics and transaction history.

| Step | What happens |
|---|---|
| 1 | Imports & global settings |
| 2 | Load data & explore it |
| 3 | Feature engineering — create smarter inputs |
| 4 | Handle missing values (imputation) |
| 5 | Create target label & split data |
| 6 | Train three models, compare results |
| 7 | Evaluate best model in depth |
| 8 | Save everything the web app needs |
| 9 | Launch the Streamlit prediction app |

> ⚠️ **Run every cell from top to bottom.** Each cell depends on the one above.

---
## Step 1 — Imports & Global Settings

We import every library and define all configuration in one place.  
Centralising settings (paths, sample size, random seed) means we only  
change one number when we want to experiment.

In [9]:
# ── Standard library ──────────────────────────────────────────────────────────
import os        # file path operations (os.path.join, os.makedirs)
import copy      # copy.deepcopy() — clone an object without affecting the original
import warnings
warnings.filterwarnings('ignore')   # suppress long sklearn convergence messages

# ── Numerical computing ───────────────────────────────────────────────────────
import numpy as np      # arrays and maths operations
import pandas as pd     # DataFrames — labelled 2-D tables

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')           # non-interactive backend — safe in Jupyter & scripts
import matplotlib.pyplot as plt
import seaborn as sns           # statistical plots built on matplotlib

# ── Machine learning (scikit-learn) ───────────────────────────────────────────
from sklearn.impute          import SimpleImputer            # fill missing values
from sklearn.model_selection import train_test_split         # split data
from sklearn.linear_model    import LogisticRegression       # Model A
from sklearn.ensemble        import RandomForestClassifier   # Model B
from sklearn.ensemble        import GradientBoostingClassifier  # Model C
from sklearn.metrics         import (
    accuracy_score,          # % of correct predictions
    roc_auc_score,           # area under the ROC curve
    confusion_matrix,        # actual vs predicted grid
    classification_report,   # precision / recall / F1 per class
)
import joblib   # save/load Python objects to disk efficiently

# ── Global configuration ──────────────────────────────────────────────────────
RANDOM_STATE = 42               # seed → same seed = same results every run
SAMPLE_SIZE  = 10_000           # rows for development  (full dataset = 1 million)
OUTPUT_DIR   = 'outputs'        # folder for plots and model files
DATA_PATH    = 'Dataset/20260411/customer_spending_1M_2018_2025.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)   # create folder if it doesn't exist

print('✅ All libraries imported.')
print(f'   Output folder: {os.path.abspath(OUTPUT_DIR)}')

✅ All libraries imported.
   Output folder: /Users/salmansakib/Documents/Projects/interview-Prep-Case-Study/module 1/outputs


---
## Step 2 — Load Data & Exploratory Data Analysis (EDA)

Before building any model we **look** at the data.  
EDA answers: What columns do we have? Any missing values? What does the target look like?

**Why sample?**  
The full 1 M-row file takes minutes. A 10 000-row sample trains in seconds  
and gives the same insights for development.

In [10]:
# ── Load the full CSV from disk ───────────────────────────────────────────────
# pd.read_csv reads a comma-separated file into a DataFrame
assert os.path.exists(DATA_PATH), f'File not found: {DATA_PATH}'
df_full = pd.read_csv(DATA_PATH)
print(f'Full dataset shape: {df_full.shape}  (rows, columns)')

# ── Take a reproducible random sample ─────────────────────────────────────────
# random_state → same rows every run  |  reset_index → clean 0-based index
df = df_full.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'Working sample shape: {df.shape}')

# ── Column names and data types ───────────────────────────────────────────────
# 'object' = text  |  'float64' = decimal  |  'int64' = integer
print('\n--- Column data types ---')
print(df.dtypes)

# ── Missing values ────────────────────────────────────────────────────────────
# NaN = Not a Number — a placeholder for absent data
print('\n--- Missing values per column ---')
print(df.isnull().sum())

# ── Descriptive statistics ────────────────────────────────────────────────────
# count / mean / std / min / quartiles / max for every numeric column
print('\n--- Numeric summary statistics ---')
print(df.describe())

# ── Histogram: Amount_spent distribution ─────────────────────────────────────
# A histogram shows how many rows fall in each spending range
plt.figure(figsize=(9, 4))
plt.hist(df['Amount_spent'].dropna(), bins=50, color='steelblue', edgecolor='white')
plt.title('Amount Spent — Distribution across the sample')
plt.xlabel('Amount Spent (৳)')
plt.ylabel('Number of transactions')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/amount_dist.png', dpi=120)
plt.show()

print('\n✅ EDA complete.')

Full dataset shape: (1000000, 11)  (rows, columns)
Working sample shape: (10000, 11)

--- Column data types ---
Transaction_ID        int64
Transaction_date        str
Gender                  str
Age                 float64
Marital_status          str
State_names             str
Segment                 str
Employees_status        str
Payment_method          str
Referral            float64
Amount_spent        float64
dtype: object

--- Missing values per column ---
Transaction_ID        0
Transaction_date      0
Gender              104
Age                 163
Marital_status        0
State_names           0
Segment               0
Employees_status    109
Payment_method        0
Referral            616
Amount_spent        958
dtype: int64

--- Numeric summary statistics ---
       Transaction_ID          Age     Referral  Amount_spent
count    1.000000e+04  9837.000000  9384.000000   9042.000000
mean     4.996075e+05    47.100335     0.644075   1409.336702
std      2.897248e+05    18.0337

---
## Step 3 — Feature Engineering

Raw columns like `Transaction_date` (a string) or `Segment` (a word) can't go  
directly into a numeric ML model. Feature engineering converts and enriches them.

| New column | Source | Why it helps |
|---|---|---|
| `amount_spent_segment` | Amount_spent | which spending quartile? |
| `spending_per_age` | Amount_spent ÷ Age | relative spend per year of life |
| `segment_target_encoded` | Segment text | mean spend of that segment (a number) |
| `days_since_start` | Transaction_date | captures time trend |
| `transaction_year/month/dayofweek` | Transaction_date | seasonality signals |
| `is_weekend` | dayofweek | weekend vs weekday behaviour |
| `month_sin / month_cos` | transaction_month | circular month encoding |

**Key principle — No Data Leakage:**  
Every threshold/mean we learn must come from **training data only**.  
We store these learned values in `fitted_params` so `app.py` can apply  
the exact same transformations at prediction time.

In [11]:
# ── Parse Transaction_date from string to real datetime ──────────────────────
# Without this, date arithmetic will raise a TypeError
# errors='coerce' turns unparseable dates into NaT instead of crashing
df['Transaction_date'] = pd.to_datetime(df['Transaction_date'], errors='coerce')
print(f'Transaction_date dtype after parsing: {df["Transaction_date"].dtype}')

# ═══════════════════════════════════════════════════════════════════════════════
# PART A — LEARN parameters from the training sample
# These get stored in fitted_params and saved to disk in Step 8.
# app.py loads this dict to reproduce the EXACT same transformations.
# ═══════════════════════════════════════════════════════════════════════════════
fitted_params = {}   # empty dict — fill it as we go

# Learn the 25th / 50th / 75th percentile of Amount_spent
# These become the boundary edges between spending bins
q25 = df['Amount_spent'].quantile(0.25)
q50 = df['Amount_spent'].quantile(0.50)   # the median
q75 = df['Amount_spent'].quantile(0.75)
fitted_params['q25'] = q25
fitted_params['q50'] = q50
fitted_params['q75'] = q75
print(f'Spending quantiles  Q25={q25:.2f}  Q50={q50:.2f}  Q75={q75:.2f}')

# Learn the average Amount_spent for each customer Segment (target encoding)
# groupby splits rows by Segment, .mean() averages Amount_spent in each group
# .to_dict() → {'Basic': 1234.5, 'Gold': 2100.0, ...}
segment_means = df.groupby('Segment')['Amount_spent'].mean().to_dict()
fitted_params['segment_means'] = segment_means
print(f'Segment means: {segment_means}')

# Learn the earliest transaction date — anchor for days_since_start
min_date = df['Transaction_date'].min()
fitted_params['min_date'] = min_date
print(f'Earliest date: {min_date}')

# Learn clipping bounds for the spending-per-age ratio
# These prevent extreme values from appearing in the app that the model never saw
age_min          = df['Age'].dropna().min()
amount_spent_max = df['Amount_spent'].dropna().max()
fitted_params['age_min']          = age_min
fitted_params['amount_spent_max'] = amount_spent_max
print(f'Min age: {age_min}  |  Max Amount_spent: {amount_spent_max:.2f}')

# ═══════════════════════════════════════════════════════════════════════════════
# PART B — APPLY transformations — add new columns to the DataFrame
# ═══════════════════════════════════════════════════════════════════════════════

# -- Feature 1: Spending bin --------------------------------------------------
# pd.cut() slices Amount_spent into 4 labelled buckets using the boundary edges
# bins = [0, q25, q50, q75, infinity] defines 4 intervals:
#   (0 – q25]   → Low_Spender
#   (q25 – q50] → Mid_Spender
#   (q50 – q75] → Upper-Mid_Spender
#   (q75 – ∞)   → High_Spender
bins   = [0, q25, q50, q75, float('inf')]
labels = ['Low_Spender', 'Mid_Spender', 'Upper-Mid_Spender', 'High_Spender']
df['amount_spent_segment'] = pd.cut(
    df['Amount_spent'],
    bins=bins,
    labels=labels,
    include_lowest=True   # ensures the minimum value falls in the first bin
)
print(f'\nSpending segment counts:\n{df["amount_spent_segment"].value_counts().to_string()}')

# -- Feature 2: Spend-per-age ratio -------------------------------------------
# Captures 'relative generosity': a 20-year-old spending ৳2000 is very different
# from a 60-year-old spending the same amount
# + 1e-6 prevents division by zero if Age happens to be 0
df['spending_per_age'] = df['Amount_spent'] / (df['Age'] + 1e-6)
# Clip: any ratio above the training maximum is capped
# This prevents the app from producing ratios the model was never trained on
max_ratio = amount_spent_max / age_min if age_min > 0 else float('inf')
df['spending_per_age'] = df['spending_per_age'].clip(upper=max_ratio)

# -- Feature 3: Segment target encoding ---------------------------------------
# Replace each Segment label ('Basic', 'Gold', …) with that segment's mean spend
# .map() looks up each row's Segment in the segment_means dictionary
# .fillna(0.0) handles any segment not seen during training
df['segment_target_encoded'] = (
    df['Segment'].map(segment_means).astype(float).fillna(0.0)
)

# -- Feature 4: Days since earliest training date -----------------------------
# Subtracting two datetimes gives a timedelta; .dt.days extracts the integer
# Captures whether spending trends have changed over time
df['days_since_start'] = (df['Transaction_date'] - min_date).dt.days

# -- Feature 5: Calendar decomposition ----------------------------------------
# Breaking a date into year / month / day-of-week lets the model learn
# from annual growth, seasonal patterns, and weekly rhythms separately
df['transaction_year']      = df['Transaction_date'].dt.year
df['transaction_month']     = df['Transaction_date'].dt.month
df['transaction_dayofweek'] = df['Transaction_date'].dt.dayofweek   # 0=Mon, 6=Sun
df['is_weekend']            = (df['transaction_dayofweek'] >= 5).astype(int)
# astype(int) converts True → 1 and False → 0 (numeric models need numbers)

# -- Feature 6: Cyclical month encoding ---------------------------------------
# Problem: month 12 (December) and month 1 (January) are adjacent on a calendar
# but the numbers 12 and 1 are far apart — the model sees a big jump.
# Fix: project each month onto a unit circle → Dec wraps smoothly to Jan.
# sin and cos together encode both direction and distance on the circle.
df['month_sin'] = np.sin(2 * np.pi * df['transaction_month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['transaction_month'] / 12)

print('\n--- Sample of engineered features (first 5 rows) ---')
new_cols = [
    'amount_spent_segment', 'spending_per_age', 'segment_target_encoded',
    'days_since_start', 'transaction_year', 'transaction_month',
    'transaction_dayofweek', 'is_weekend', 'month_sin', 'month_cos'
]
print(df[new_cols].head(5).to_string())
print('\n✅ Feature engineering complete.')

Transaction_date dtype after parsing: datetime64[us]
Spending quantiles  Q25=674.40  Q50=1324.12  Q75=2036.57
Segment means: {'Basic': 1400.7046778100776, 'Gold': 1374.3961761546725, 'Missing': 1442.0132890855457, 'Platinum': 1428.2105189873419, 'Silver': 1418.7206260869566}
Earliest date: 2018-01-01 04:32:00
Min age: 15.0  |  Max Amount_spent: 2999.98

Spending segment counts:
amount_spent_segment
Upper-Mid_Spender    2263
Low_Spender          2261
Mid_Spender          2261
High_Spender         2257

--- Sample of engineered features (first 5 rows) ---
  amount_spent_segment  spending_per_age  segment_target_encoded  days_since_start  transaction_year  transaction_month  transaction_dayofweek  is_weekend     month_sin  month_cos
0          Mid_Spender         29.907812             1442.013289              2553              2024                 12                      6           1 -2.449294e-16   1.000000
1          Low_Spender         21.452272             1400.704678               2

---
## Step 4 — Handle Missing Values (Imputation)

Most ML models crash with `NaN` values. We must fill every gap.

**Strategy: mean imputation**  
Replace each `NaN` with the **column average** from training data.  
Simple, interpretable, and usually effective.

**Critical rule:** we `fit` the imputer on training data **once**,  
then at prediction time we `transform` with those stored means — never recompute.

In [12]:
# ── Choose input features (X columns) ────────────────────────────────────────
# We deliberately EXCLUDE:
#   'Amount_spent'         → this is our TARGET label, not an input
#   'Transaction_ID'       → a unique row ID with zero predictive signal
#   'amount_spent_segment' → categorical text bins; its signal is already
#                            captured by 'spending_per_age' and 'segment_target_encoded'
FEATURE_COLS = [
    'Age',                       # customer age in years
    'Referral',                  # 1.0 = referred by a friend, 0.0 = not
    'spending_per_age',          # Amount_spent / Age (engineered Step 3)
    'segment_target_encoded',    # mean spend for this segment (engineered Step 3)
    'days_since_start',          # days from earliest date (engineered Step 3)
    'transaction_year',          # calendar year
    'transaction_month',         # calendar month (1–12)
    'transaction_dayofweek',     # day of week (0=Mon … 6=Sun)
    'is_weekend',                # 1 = Sat/Sun, 0 = weekday
    'month_sin',                 # cyclical month — sine component
    'month_cos',                 # cyclical month — cosine component
]

# Store the column list so app.py uses the exact same order
fitted_params['feature_cols'] = FEATURE_COLS

# Extract only the chosen columns
X_raw = df[FEATURE_COLS].copy()

print('--- Missing values BEFORE imputation ---')
missing_before = X_raw.isnull().sum()
print(missing_before[missing_before > 0].to_string() or '  (none)')
print(f'Total NaNs: {missing_before.sum()}')

# ── Fit the imputer ───────────────────────────────────────────────────────────
# strategy='mean' → learn the average of each column from X_raw
# fit_transform() = .fit() then .transform() in one step
#   .fit()      → compute and store the mean of every column
#   .transform()→ replace NaN values with those stored means
imputer = SimpleImputer(strategy='mean')
X_imputed_array = imputer.fit_transform(X_raw)
# Result is a NumPy array; we convert it back to a DataFrame for readability
X = pd.DataFrame(X_imputed_array, columns=FEATURE_COLS, index=df.index)

print('\n--- Missing values AFTER imputation ---')
missing_after = X.isnull().sum()
print(missing_after[missing_after > 0].to_string() or '  (none — all NaNs filled ✅)')

print(f'\nFinal feature matrix: {X.shape[0]} rows × {X.shape[1]} columns')
print(f'Columns: {list(X.columns)}')
print('\n✅ Imputation complete. X is clean and model-ready.')

--- Missing values BEFORE imputation ---
Age                  163
Referral             616
spending_per_age    1116
Total NaNs: 1895

--- Missing values AFTER imputation ---
Series([], )

Final feature matrix: 10000 rows × 11 columns
Columns: ['Age', 'Referral', 'spending_per_age', 'segment_target_encoded', 'days_since_start', 'transaction_year', 'transaction_month', 'transaction_dayofweek', 'is_weekend', 'month_sin', 'month_cos']

✅ Imputation complete. X is clean and model-ready.


---
## Step 5 — Create Target Variable & Train/Test Split

**Target variable:** convert continuous `Amount_spent` into 3 classes.

| Class | Range | Share |
|---|---|---|
| Low | ≤ 33rd percentile | ≈ 33 % |
| Medium | 33rd – 67th percentile | ≈ 34 % |
| High | > 67th percentile | ≈ 33 % |

**Why percentiles instead of fixed ৳ amounts?**  
Percentile thresholds automatically balance class sizes whatever the data range is.

**Train/test split:**  
20 % of rows are held out as a **test set** the model never sees during training.  
`stratify=y` ensures every class is proportionally represented in both halves.

In [13]:
# ── Compute percentile thresholds for spending classes ────────────────────────
# quantile(0.33) = value below which 33% of Amount_spent values fall
spend_q33 = df['Amount_spent'].quantile(0.33)
spend_q67 = df['Amount_spent'].quantile(0.67)
print(f'Low  / Medium boundary: {spend_q33:.2f}')
print(f'Medium / High boundary: {spend_q67:.2f}')

# ── Assign a class label to every row ─────────────────────────────────────────
# np.select checks conditions in order — the first True condition wins
# If Amount_spent <= spend_q33            → label 'Low'
# Else if Amount_spent <= spend_q67       → label 'Medium'
# Else (neither condition matched)        → label 'High'  (default)
conditions = [
    df['Amount_spent'] <= spend_q33,
    df['Amount_spent'] <= spend_q67,
]
choices = ['Low', 'Medium']
y = np.select(conditions, choices, default='High')

print('\nClass distribution:')
for cls, cnt in pd.Series(y).value_counts().items():
    print(f'   {cls}: {cnt} rows  ({cnt / len(y) * 100:.1f}%)')

# ── Split into training (80%) and test (20%) sets ─────────────────────────────
# test_size=0.2     → 20% goes to test
# random_state      → reproducible split
# stratify=y        → same class proportions in train AND test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'\nTraining set : {X_train.shape[0]} rows  ({X_train.shape[0] / len(X) * 100:.0f}%)')
print(f'Test set     : {X_test.shape[0]} rows  ({X_test.shape[0] / len(X) * 100:.0f}%)')
print(f'Features     : {X_train.shape[1]} columns')
print('\n✅ Data split complete.')

Low  / Medium boundary: 880.28
Medium / High boundary: 1799.57

Class distribution:
   High: 3942 rows  (39.4%)
   Medium: 3074 rows  (30.7%)
   Low: 2984 rows  (29.8%)

Training set : 8000 rows  (80%)
Test set     : 2000 rows  (20%)
Features     : 11 columns

✅ Data split complete.


---
## Step 6 — Train Three Models & Compare

We train three algorithms on the **same** training data and evaluate each  
on the **same** test data — a fair apples-to-apples comparison.

| Algorithm | Core idea |
|---|---|
| **Logistic Regression** | Finds a linear decision boundary; fast and interpretable |
| **Random Forest** | Votes from 100 independent trees; robust to noise |
| **Gradient Boosting** | Trees built sequentially; each fixes previous errors |

**Primary metric: ROC-AUC (macro)**  
AUC = 1.0 → perfect · AUC = 0.5 → random guessing.  
Macro averaging: compute AUC per class, then take the equal-weighted average.

In [14]:
# ─── Model A: Logistic Regression ────────────────────────────────────────────
# Linear model — finds a boundary that is a straight line (or hyperplane in N-D)
# Best for: linearly separable data, fast prototyping, high interpretability
# max_iter=2000 — give the solver enough steps to converge
lr_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
lr_model.fit(X_train, y_train)              # learn from training data
lr_preds = lr_model.predict(X_test)         # predict class labels
lr_proba = lr_model.predict_proba(X_test)   # predict class probabilities for AUC
lr_acc  = accuracy_score(y_test, lr_preds)
lr_auc  = roc_auc_score(y_test, lr_proba, multi_class='ovr', average='macro')
print(f'Logistic Regression  →  Accuracy: {lr_acc:.4f}  |  ROC-AUC: {lr_auc:.4f}')

# ─── Model B: Random Forest ───────────────────────────────────────────────────
# Trains 100 decision trees on random subsets of rows AND features
# Each tree votes; the majority vote wins (bagging / bootstrap aggregation)
# Advantage: handles non-linear patterns, robust to outliers, little tuning needed
# n_jobs=-1 — use all CPU cores in parallel (faster training)
rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)
rf_acc  = accuracy_score(y_test, rf_preds)
rf_auc  = roc_auc_score(y_test, rf_proba, multi_class='ovr', average='macro')
print(f'Random Forest        →  Accuracy: {rf_acc:.4f}  |  ROC-AUC: {rf_auc:.4f}')

# ─── Model C: Gradient Boosting ──────────────────────────────────────────────
# Builds trees ONE AT A TIME; each new tree is trained to fix mistakes of previous ones
# The 'gradient' means it minimises the loss function step by step (like gradient descent)
# Usually achieves highest accuracy; slowest to train of the three
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)
gb_model.fit(X_train, y_train)
gb_preds = gb_model.predict(X_test)
gb_proba = gb_model.predict_proba(X_test)
gb_acc  = accuracy_score(y_test, gb_preds)
gb_auc  = roc_auc_score(y_test, gb_proba, multi_class='ovr', average='macro')
print(f'Gradient Boosting    →  Accuracy: {gb_acc:.4f}  |  ROC-AUC: {gb_auc:.4f}')

# ─── Summary table (sorted best → worst by ROC-AUC) ──────────────────────────
results_df = pd.DataFrame({
    'Model':    ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'Accuracy': [lr_acc, rf_acc, gb_acc],
    'ROC-AUC':  [lr_auc, rf_auc, gb_auc],
}).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print('\n=== Model Comparison (sorted best → worst by ROC-AUC) ===')
print(results_df.to_string(index=False))

# ─── Select the best model ────────────────────────────────────────────────────
# The top row after sorting has the highest ROC-AUC — that is our winner
best_model_name = results_df.iloc[0]['Model']
model_lookup = {
    'Logistic Regression': lr_model,
    'Random Forest':       rf_model,
    'Gradient Boosting':   gb_model,
}
best_model = model_lookup[best_model_name]

print(f'\n🏆  Best model: {best_model_name}  (ROC-AUC = {results_df.iloc[0]["ROC-AUC"]:.4f})')
print('\n✅ Training complete.')

Logistic Regression  →  Accuracy: 0.8110  |  ROC-AUC: 0.9198
Random Forest        →  Accuracy: 0.9620  |  ROC-AUC: 0.9966
Gradient Boosting    →  Accuracy: 0.9815  |  ROC-AUC: 0.9989

=== Model Comparison (sorted best → worst by ROC-AUC) ===
              Model  Accuracy  ROC-AUC
  Gradient Boosting    0.9815 0.998936
      Random Forest    0.9620 0.996636
Logistic Regression    0.8110 0.919775

🏆  Best model: Gradient Boosting  (ROC-AUC = 0.9989)

✅ Training complete.


---
## Step 7 — Evaluate the Best Model in Depth

A single accuracy number hides a lot. We drill deeper with:

- **Classification report** — precision, recall, F1 per class  
  - *Precision:* of all rows predicted as class X, how many were actually X?  
  - *Recall:* of all rows that ARE class X, what fraction did we predict correctly?  
  - *F1:* harmonic mean of precision and recall  
- **Confusion matrix** — which classes get confused with each other?  
- **Confidence histogram** — how certain is the model about each prediction?

In [15]:
# ── Get predictions and probabilities from the best model ─────────────────────
y_pred = best_model.predict(X_test)          # class labels: 'Low' / 'Medium' / 'High'
y_prob = best_model.predict_proba(X_test)    # shape: (n_rows, 3) — one prob per class
classes = best_model.classes_                # ['High', 'Low', 'Medium'] (alphabetical)

# ─── A: Classification Report ─────────────────────────────────────────────────
print('--- Classification Report ---')
report_text = classification_report(y_test, y_pred, target_names=classes)
print(report_text)
with open(f'{OUTPUT_DIR}/classification_report.txt', 'w') as f:
    f.write(f'Best model: {best_model_name}\n\n')
    f.write(report_text)

# ─── B: Confusion Matrix ──────────────────────────────────────────────────────
# Rows = true class  |  Columns = predicted class
# Diagonal cells = correct predictions  |  Off-diagonal = errors
cm = confusion_matrix(y_test, y_pred, labels=classes)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,            # print count inside each cell
    fmt='d',               # integer format (not scientific notation)
    cmap='Blues',          # darker = higher count
    xticklabels=classes,
    yticklabels=classes,
    linewidths=0.5,
)
plt.title(f'Confusion Matrix — {best_model_name}')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=120)
plt.show()

# ─── C: Confidence Histogram ──────────────────────────────────────────────────
# For each row the model outputs 3 probabilities summing to 1.0
# max(axis=1) picks the highest — the model's 'confidence' in its chosen class
# A trustworthy model: most predictions cluster near 1.0
max_confidence = y_prob.max(axis=1)

plt.figure(figsize=(8, 4))
plt.hist(max_confidence, bins=30, color='#2ecc71', edgecolor='white')
plt.axvline(max_confidence.mean(), color='red', linestyle='--',
            label=f'Mean confidence = {max_confidence.mean():.2f}')
plt.title(f'Prediction Confidence — {best_model_name}')
plt.xlabel('Highest class probability (model confidence)')
plt.ylabel('Number of test rows')
plt.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confidence_hist.png', dpi=120)
plt.show()

print(f'Average confidence: {max_confidence.mean():.3f}')
print('\n✅ Evaluation complete.')

--- Classification Report ---
              precision    recall  f1-score   support

        High       1.00      0.98      0.99       788
         Low       1.00      0.97      0.98       597
      Medium       0.95      1.00      0.97       615

    accuracy                           0.98      2000
   macro avg       0.98      0.98      0.98      2000
weighted avg       0.98      0.98      0.98      2000

Average confidence: 0.936

✅ Evaluation complete.


---
## Step 8 — Save All Artefacts

The web app (`app.py`) must reproduce the **exact same pipeline** at inference time.
We save three files:

| File | Contents | Why needed |
|---|---|---|
| `final_best_model.joblib` | trained classifier | makes predictions |
| `fitted_params.joblib` | dict: quantiles, means, min_date, column list | reproduce feature engineering |
| `imputer.joblib` | fitted SimpleImputer | fill NaNs using TRAINING means |

> **Why not recompute in the app?**  
> Computing thresholds from user input would give different values each time.  
> The model was trained with specific boundaries — prediction must use the same ones.

In [16]:
# ─── 8a. Retrain the best model on 100% of the sample ────────────────────────
# During comparison (Step 6) we trained on only 80% (the training split).
# Now we retrain on ALL available rows — more data = more patterns learned.
# copy.deepcopy clones the model with the SAME hyperparameters but unfitted state.
final_model = copy.deepcopy(best_model)   # same algorithm + same settings
final_model.fit(X, y)                     # fit on ALL sample rows (no hold-out)
print(f'Final model: {best_model_name}')
print(f'  Trained on: {X.shape[0]} rows  ({X.shape[1]} features)')

# ─── 8b. Save the trained model ───────────────────────────────────────────────
# joblib.dump serialises a Python object to a binary file efficiently
model_path = f'{OUTPUT_DIR}/final_best_model.joblib'
joblib.dump(final_model, model_path)
print(f'\nModel saved    → {model_path}  ({os.path.getsize(model_path):,} bytes)')

# ─── 8c. Save fitted_params ───────────────────────────────────────────────────
# This dict was built through Steps 3 & 4 and contains:
#   q25, q50, q75           → quantile edges for pd.cut() spending bins
#   segment_means           → {segment_name: mean_spend} for target encoding
#   min_date                → anchor date for days_since_start calculation
#   age_min, amount_spent_max → clipping bounds for spending_per_age
#   feature_cols            → exact column list in the correct order for imputer
params_path = f'{OUTPUT_DIR}/fitted_params.joblib'
joblib.dump(fitted_params, params_path)
print(f'Params saved   → {params_path}  ({os.path.getsize(params_path):,} bytes)')
print(f'  Keys stored: {list(fitted_params.keys())}')

# ─── 8d. Save the imputer ─────────────────────────────────────────────────────
# The imputer was fitted in Step 4; it stores the column means from training.
# In the app we call imputer.transform() — it replaces NaN with those means.
imputer_path = f'{OUTPUT_DIR}/imputer.joblib'
joblib.dump(imputer, imputer_path)
print(f'Imputer saved  → {imputer_path}  ({os.path.getsize(imputer_path):,} bytes)')
print(f'  Columns: {list(imputer.feature_names_in_)}')

print(f"\n✅ All 3 artefacts saved to '{OUTPUT_DIR}/'")
print('   Next: run  streamlit run app.py  to open the prediction app.')

Final model: Gradient Boosting
  Trained on: 10000 rows  (11 features)

Model saved    → outputs/final_best_model.joblib  (399,703 bytes)
Params saved   → outputs/fitted_params.joblib  (639 bytes)
  Keys stored: ['q25', 'q50', 'q75', 'segment_means', 'min_date', 'age_min', 'amount_spent_max', 'feature_cols']
Imputer saved  → outputs/imputer.joblib  (1,023 bytes)
  Columns: ['Age', 'Referral', 'spending_per_age', 'segment_target_encoded', 'days_since_start', 'transaction_year', 'transaction_month', 'transaction_dayofweek', 'is_weekend', 'month_sin', 'month_cos']

✅ All 3 artefacts saved to 'outputs/'
   Next: run  streamlit run app.py  to open the prediction app.


---
## Step 9 — Launch the Streamlit Prediction App

The app (`app.py`) loads the three `.joblib` files from Step 8 and provides  
an interactive form for real-time predictions.  
Run the cell to start the server and embed the UI below.

In [17]:
import subprocess, sys, time, threading
from IPython.display import display, HTML

# Start the Streamlit server in a background daemon thread
# daemon=True means the thread auto-stops when the notebook kernel shuts down
def start_server():
    subprocess.Popen([
        sys.executable, '-m', 'streamlit', 'run', 'app.py',
        '--server.headless', 'true',   # do not open a browser tab automatically
        '--server.port',    '8501',
        '--server.address', '127.0.0.1',
    ])

print('Starting Streamlit on http://127.0.0.1:8501 ...')
threading.Thread(target=start_server, daemon=True).start()
time.sleep(4)   # give the server time to boot before the iframe loads

# Embed the running app as an iframe directly inside this notebook cell
display(HTML("""<iframe src="http://127.0.0.1:8501" width="100%" height="800px" style="border:1px solid #ddd;border-radius:8px;"></iframe>"""))
print('App is live above ☝️  Interact with the form to get predictions.')
print('To stop: restart the kernel, or kill the subprocess manually.')

Starting Streamlit on http://127.0.0.1:8501 ...


2026-04-16 22:10:17.134 Port 8501 is not available


App is live above ☝️  Interact with the form to get predictions.
To stop: restart the kernel, or kill the subprocess manually.
